# Topic embeddings and comment means in one t-SNE projection

Circles show SGNS topic embeddings and diamonds show mean comment embeddings. A thin line joins the two representations of the same full topic path. Unpaired topics appear as individual points.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import qualitative
from scipy.linalg import orthogonal_procrustes
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "topic_embeddings").is_dir())
TOPIC_KEY = ["Topic_1", "Topic_2", "Topic_3"]
SEED = 42
PERPLEXITY = 20

topics = pd.read_csv(ROOT / "results/sgns/topic_embeddings_normalized.csv", keep_default_na=False)
means = pd.read_csv(ROOT / "data/processed/comment_embedding_means.csv", keep_default_na=False)
pairs = topics[TOPIC_KEY].reset_index(names="topic_row").merge(
    means[TOPIC_KEY].reset_index(names="mean_row"), on=TOPIC_KEY, validate="one_to_one"
)
print(f"{len(topics):,} topic embeddings, {len(means):,} comment means, {len(pairs):,} matched paths")

1,000 topic embeddings, 2,077 comment means, 1,000 matched paths


## Align on shared topic paths

1. **Normalize each vector.** Divide each topic embedding and comment mean by its own length so each starts with length 1.
2. **Center the topics and reduce the comment means.** Subtract the average matched topic vector from every topic vector, keeping all 150 dimensions. Fit PCA on the matched comment means and use it to project all comment means from 896 to 150 dimensions.
3. **Put the spaces on the same scale.** In each space, compute the root mean square (RMS) length of its matched vectors. Divide every vector in that space by this single number so the matched vectors have an RMS length of 1.
4. **Align the comment means to the topics.** Use the matched paths to fit one orthogonal transformation (a rotation or reflection) that minimizes the sum of squared distances between paired vectors. Apply it to every comment mean.

In [2]:
topic_vectors = topics.filter(regex=r"^embedding_\d+$").to_numpy(dtype=float)
mean_vectors = means.filter(regex=r"^dim_\d+$").to_numpy(dtype=float)
topic_rows = pairs["topic_row"].to_numpy()
mean_rows = pairs["mean_row"].to_numpy()
dims = topic_vectors.shape[1]
if dims > min(len(pairs) - 1, mean_vectors.shape[1]):
    raise ValueError(f"Too few matched paths or comment dimensions to fit {dims} PCA components.")

topic_coords = normalize(topic_vectors)
topic_coords -= topic_coords[topic_rows].mean(axis=0)

unit_means = normalize(mean_vectors)
mean_pca = PCA(n_components=dims, svd_solver="full").fit(unit_means[mean_rows])
mean_coords = mean_pca.transform(unit_means)

def scale_to_matched_rms(coords, rows):
    """Scale coordinates to have the same root mean square as the matched vectors."""
    rms = np.sqrt(np.square(coords[rows]).sum(axis=1).mean())
    if rms == 0:
        raise ValueError("Matched vectors have no variation.")
    return coords / rms

topic_coords = scale_to_matched_rms(topic_coords, topic_rows)
mean_coords = scale_to_matched_rms(mean_coords, mean_rows)
rotation, _ = orthogonal_procrustes(mean_coords[mean_rows], topic_coords[topic_rows])
aligned_means = mean_coords @ rotation

combined = np.vstack([topic_coords, aligned_means])
xy = TSNE(n_components=2, perplexity=PERPLEXITY, init="pca", learning_rate="auto",
          max_iter=1000, random_state=SEED).fit_transform(combined)

## Explore the projection

Hover over a point for its path and source. Click a main topic in the legend to toggle both sets of points and their connecting lines.

In [3]:
sources = {"Topic embedding": "circle", "Comment mean": "diamond"}
points = pd.concat(
    [topics[TOPIC_KEY].assign(source="Topic embedding"), means[TOPIC_KEY].assign(source="Comment mean")],
    ignore_index=True,
)
points[["x", "y"]] = xy
points["path"] = points[TOPIC_KEY].apply(lambda row: " / ".join(filter(None, row)), axis=1)

fig = go.Figure()
for main_topic, group in pairs.groupby("Topic_1", sort=False):
    starts = xy[group["topic_row"]]
    ends = xy[len(topics) + group["mean_row"]]
    # NaN separators keep adjacent pairs from being joined.
    segments = np.stack([starts, ends, np.full_like(starts, np.nan)], axis=1).reshape(-1, 2)
    fig.add_scatter(x=segments[:, 0], y=segments[:, 1], mode="lines",
                    line=dict(color="rgba(75,75,75,0.20)", width=0.6),
                    hoverinfo="skip", legendgroup=main_topic, showlegend=False)

palette = qualitative.Plotly + qualitative.Dark24
for i, (main_topic, group) in enumerate(points.groupby("Topic_1", sort=False)):
    color = palette[i % len(palette)]
    for j, (source, rows) in enumerate(group.groupby("source", sort=False)):
        fig.add_scatter(
            x=rows.x, y=rows.y, mode="markers", name=main_topic, legendgroup=main_topic,
            showlegend=(j == 0), marker=dict(symbol=sources[source], color=color, size=7, opacity=0.8),
            text=rows.path, hovertemplate="%{text}<br>" + source + "<extra></extra>",
        )

fig.update_layout(
    title="Topic embeddings (circles) and comment means (diamonds)",
    xaxis_title="t-SNE 1", yaxis_title="t-SNE 2", legend_title_text="Main topic",
    legend_groupclick="togglegroup", template="plotly_white", height=800,
)
output = ROOT / "output/topic_comment_means_joint_tsne.html"
output.parent.mkdir(exist_ok=True)
fig.write_html(output)
fig.show()

In [4]:
# Top 10 neighbors of lifestyle / raetsel-sudoku / sudoku
from IPython.display import display

sudoku_path = ("lifestyle", "raetsel-sudoku", "sudoku")
match = pairs.set_index(TOPIC_KEY).loc[sudoku_path]
topic_idx, mean_idx = int(match["topic_row"]), int(match["mean_row"])
is_sudoku = points[TOPIC_KEY].eq(sudoku_path, axis="columns").all(axis=1).to_numpy(copy=True)

def show_neighbors(title, candidates, distances, exclude):
    table = candidates[["path", "source"]].assign(distance=distances)
    table = table.loc[~exclude].nsmallest(10, "distance").reset_index(drop=True)
    table.index = pd.RangeIndex(1, len(table) + 1, name="Rank")
    print(title)
    display(table.round({"distance": 4}))

# Rank by cosine distance in each original space. Smaller values mean closer neighbors.
for title, vectors, query_idx, rows in [
    ("Topic space · cosine distance", topic_vectors, topic_idx, slice(0, len(topics))),
    ("Comment mean space · cosine distance", mean_vectors, mean_idx, slice(len(topics), None)),
]:
    unit = normalize(vectors)
    distances = np.clip(1 - unit @ unit[query_idx], 0, 2)
    show_neighbors(title, points.iloc[rows], distances, is_sudoku[rows])

# Query both Sudoku points against all aligned vectors using the joint t-SNE input metric.
for source, query_idx in [("topic embedding", topic_idx), ("comment mean", len(topics) + mean_idx)]:
    distances = np.linalg.norm(combined - combined[query_idx], axis=1)
    show_neighbors(f"Shared space from Sudoku {source} · Euclidean distance",
                   points, distances, is_sudoku)

Topic space · cosine distance


,path,source,distance
Rank,,,
1,diskurs / blogs / quizblog,Topic embedding,0.5333
2,diskurs / offtopic,Topic embedding,0.5514
3,web / games / gewinnspielegames,Topic embedding,0.5718
4,diskurs / userblogs / quizblog,Topic embedding,0.5767
5,sport / fussball / werderbremen,Topic embedding,0.5891
6,zukunft / ideen-fuer-eine-bessere-welt,Topic embedding,0.5950
7,web / webmix / webquiz,Topic embedding,0.5954
8,kultur / literatur / literaturforumuser,Topic embedding,0.6046
9,panorama / oesterreich / laenderchronik,Topic embedding,0.6083


Comment mean space · cosine distance


,path,source,distance
Rank,,,
1,lifestyle / raetsel-sudoku,Comment mean,0.0001
2,diskurs / offtopic,Comment mean,0.1465
3,diskurs / userblogs / quizblog,Comment mean,0.1491
4,lifestyle / raetsel-sudoku / kreuzwortraetsel,Comment mean,0.1497
5,diskurs / blogs / quizblog,Comment mean,0.1507
6,diskurs / mitraten-quiz,Comment mean,0.1507
7,diskurs / blogs / quizblogopenscience,Comment mean,0.1536
8,sport / mehrsport / bilddestages,Comment mean,0.1558
9,sport / mehrsport / tischtennis1,Comment mean,0.1587


Shared space from Sudoku topic embedding · Euclidean distance


,path,source,distance
Rank,,,
1,diskurs,Comment mean,1.0044
2,inland / oesterreichwaehlt,Comment mean,1.0585
3,podcast,Comment mean,1.0598
4,lifestyle,Comment mean,1.0726
5,video,Comment mean,1.0821
6,diskurs / mitreden,Comment mean,1.0858
7,panorama / gesellschaft / datenpunkt,Comment mean,1.0889
8,panorama,Comment mean,1.0904
9,podcast / standardhoeren,Comment mean,1.0966


Shared space from Sudoku comment mean · Euclidean distance


,path,source,distance
Rank,,,
1,lifestyle / raetsel-sudoku,Comment mean,0.0561
2,lifestyle / raetsel-sudoku / kreuzwortraetsel,Comment mean,2.2457
3,diskurs / offtopic,Comment mean,2.2823
4,diskurs / userblogs / quizblog,Comment mean,2.3106
5,diskurs / mitraten-quiz,Comment mean,2.3211
6,diskurs / blogs / quizblog,Comment mean,2.3227
7,diskurs / blogs / quizblogopenscience,Comment mean,2.3243
8,sport / mehrsport / bilddestages,Comment mean,2.3550
9,web / games / wirentwickelneinspiel,Comment mean,2.3551


## Closest and farthest topics from their own comment means

For each matched topic path, measure the Euclidean distance between its SGNS vector and its aligned comment mean before the 2D t-SNE projection. Smaller distances indicate closer agreement after fitting the alignment on these same paths.

In [5]:
pair_distances = pairs[TOPIC_KEY].copy()
pair_distances["distance"] = np.linalg.norm(
    topic_coords[topic_rows] - aligned_means[mean_rows], axis=1
)

for title, table in [
    ("15 closest to their own comment mean", pair_distances.nsmallest(15, "distance")),
    ("15 farthest from their own comment mean", pair_distances.nlargest(15, "distance")),
]:
    table = table.reset_index(drop=True)
    table.index = pd.RangeIndex(1, len(table) + 1, name="Rank")
    print(title)
    display(table.round({"distance": 4}))

15 closest to their own comment mean


,Topic_1,Topic_2,Topic_3,distance
Rank,,,,
1,sport,fussball,fussballinternational,0.4756
2,web,games,gamepolitics,0.5129
3,panorama,gesellschaft,panoramatiere,0.5590
4,international,asien,china,0.5765
5,international,amerika,usa,0.5768
6,wissenschaft,zeit,archaeologie,0.5867
7,kultur,film,filmkritik,0.5895
8,etat,tv,tvserien,0.6041
9,international,europa,grossbritannien,0.6144


15 farthest from their own comment mean


,Topic_1,Topic_2,Topic_3,distance
Rank,,,,
1,lifestyle,raetsel-sudoku,sudoku,1.9225
2,web,games,gewinnspielegames,1.2498
3,web,innovationen,phone7,1.2143
4,sport,fussball,superleague,1.2118
5,diskurs,mitreden,leben-mit-tieren,1.2113
6,zukunft,alternative-antriebe,,1.1469
7,diskurs,offtopic,,1.1465
8,immobilien,immogewerbe,immowirtschaft,1.1462
9,etat,sosindwir,,1.1057


## Pair recovery among the top 1, 3 and 5 neighbors

**Nearest-comment-mean pair.** For each shared path A, find the other shared path B with the most similar comment mean. Count A → B as recovered if B is among A’s top *k* topic embedding neighbors. Both searches use cosine similarity in their respective original spaces and exclude A itself. Each query contributes one directed pair, so reciprocal neighbors count once per direction.

**Own comment mean.** For each matched topic embedding, search all comment means by Euclidean distance in the aligned space and check whether its own path appears in the top *k*. This measures retrieval on the paths used to fit the alignment. Both checks use the coordinates before t-SNE and break ties by input order.

In [6]:
from scipy.spatial.distance import cdist

def cosine_neighbors(vectors, k):
    unit = normalize(vectors)
    scores = unit @ unit.T
    np.fill_diagonal(scores, -np.inf)
    return np.argsort(-scores, axis=1, kind="stable")[:, :k]

# Both arrays follow the same matched-path order, so neighbor indices identify paths.
mean_neighbor = cosine_neighbors(mean_vectors[mean_rows], 1)
topic_neighbors = cosine_neighbors(topic_vectors[topic_rows], 5)
distances_to_means = cdist(topic_coords[topic_rows], aligned_means)
nearest_means = np.argsort(distances_to_means, axis=1, kind="stable")[:, :5]

recovery = []
for label, hits in [
    ("Nearest-comment-mean pair", topic_neighbors == mean_neighbor),
    ("Own comment mean", nearest_means == mean_rows[:, None]),
]:
    for k in (1, 3, 5):
        count = int(hits[:, :k].any(axis=1).sum())
        recovery.append([label, k, count, len(pairs), 100 * count / len(pairs)])

display(pd.DataFrame(recovery, columns=["Pair", "Top k", "Recovered", "Queries", "Recovery (%)"])
        .set_index(["Pair", "Top k"]).round(2))

Recovered  Queries  Recovery (%)
Pair                      Top k                                  
Nearest-comment-mean pair 1            318     1000          31.8
                          3            518     1000          51.8
                          5            618     1000          61.8
Own comment mean          1            664     1000          66.4
                          3            851     1000          85.1
                          5            901     1000          90.1